In [ ]:
from reachy_sdk import ReachySDK

Connect to Reachy. If you're working directly on the robot, use

```python
host='localhost'
```

If not, just get Reachy's IP address with

```bash
$ ifconfig
```

and replace the host argument zith the IP address.

In [ ]:
reachy = ReachySDK(host='10.22.129.133')

Check if you see the five joints of the head.

In [ ]:
reachy.head

Use look_at to see if the zero has been actually done.

In [ ]:
# 1. Turn stiffness back ON so the motors can move
from reachy_sdk.trajectory import goto

# 1. Re-engage head motor stiffness
reachy.head.compliant = False

# 2. Extract the true joint variables from the head.joints attribute
head_center_pose = {
    reachy.head.joints.neck_roll: 0,
    reachy.head.joints.neck_pitch: 0,
    reachy.head.joints.neck_yaw: 0,
}

# 3. Trigger the movement
goto(
    goal_positions=head_center_pose,
    duration=2.0
)

In [ ]:
reachy.head.joints.neck_roll.goal_position = 0
reachy.head.joints.neck_pitch.goal_position = 0
reachy.head.joints.neck_yaw.goal_position = 0

In [ ]:
#look_down = reachy.head.look_at(
#    x=0.3,
#    y=0.0,
#    z=-0.4,  #0.4
#    duration=1.0,
#    starting_positions={
#        reachy.head.neck_roll: reachy.head.neck_roll.goal_position,
#        reachy.head.neck_pitch: reachy.head.neck_pitch.goal_position,
#        reachy.head.neck_yaw: reachy.head.neck_yaw.goal_position
#    })

In [ ]:
import time

import cv2

# Loop to smoothly raise Z from -0.4 to 0.0
for z_height in [-0.6, -0.5, -0.4, -0.3, 0.0]:
    reachy.head.look_at(0.5, 0.0, z_height, duration=1.0)
   
    # 1. Grab frames from both cameras
    left_frame = reachy.left_camera.last_frame
    right_frame = reachy.right_camera.last_frame

    # 2. Check Left Camera status
    if left_frame is not None and left_frame.size > 0:
        print("✅ Left camera is ONLINE. Shape:", left_frame.shape)
        cv2.imshow("Left Camera", left_frame)
    else:
        print("❌ Left camera is OFFLINE or returning empty frames.")

    # 3. Check Right Camera status
    if right_frame is not None and right_frame.size > 0:
        print("✅ Right camera is ONLINE. Shape:", right_frame.shape)
        cv2.imshow("Right Camera", right_frame)
    else:
        print("❌ Right camera is OFFLINE or returning empty frames.")

    # Keep the window open until you press any key
    if (left_frame is not None) or (right_frame is not None):
        print("Displaying active feeds. Press any key on the image window to close.")
        cv2.waitKey(0)
        
    time.sleep(2.0)  # Pause for 1 second at each step

In [ ]:
reachy.turn_off('head')